# LeetCode 2052 — Minimum Cost to Separate Sentence Into Rows bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-to-separate-sentence-into-rows/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

> **Lưu ý: đây là bài Premium** trên LeetCode — cần tài khoản Premium để xem đề gốc và nộp bài. Đề dưới đây viết theo hiểu biết về bài (nguồn: bản đề của doocs/leetcode); nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc**.

Skeleton LeetCode:

```python
class Solution:
    def minimumCost(self, sentence: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa bài "**ngắt dòng văn bản**" (word wrap) thành **quy hoạch động trên hậu tố**: `dp[i]` = chi phí nhỏ nhất để xếp các từ `i..m−1`.
2. Xử lý điểm đặc biệt: dòng **cuối cùng không tính chi phí** ⇒ nếu mọi từ còn lại vừa một dòng thì `dp[i] = 0`.
3. Tính độ dài một dòng (gồm **khoảng trắng** giữa các từ) bằng cộng dồn hoặc tổng tiền tố.
4. Thấy vì sao **tham lam** (nhồi đầy mỗi dòng) **không** tối ưu với chi phí bình phương.
5. Đối chiếu bằng vét cạn mọi cách ngắt (`2^(m−1)`).

## 2. Đề bài & ví dụ

Cho chuỗi `sentence` gồm các từ cách nhau **một** khoảng trắng (không có khoảng trắng đầu/cuối) và số nguyên `k`. Tách câu thành các **dòng**, mỗi dòng dài **tối đa** `k` ký tự, bằng cách chèn ngắt dòng **giữa** các từ: không được cắt một từ, giữ nguyên thứ tự, các từ trên cùng dòng cách nhau đúng một khoảng trắng, dòng không bắt đầu/kết thúc bằng khoảng trắng.

**Chi phí** một dòng dài `n` là `(k − n)²`; tổng chi phí là tổng chi phí mọi dòng **trừ dòng cuối**. Trả về tổng chi phí nhỏ nhất.

**Ví dụ 1:** `"i love leetcode"`, `k = 12` → `36` (`"i love" | "leetcode"`: `(12 − 6)² = 36`; cách `"i" | "love" | "leetcode"` tốn `121 + 64 = 185`; `"love leetcode"` dài 13 > 12 nên không được).

**Ví dụ 2:** `"apples and bananas taste great"`, `k = 7` → `21` (mỗi từ một dòng: `1 + 16 + 0 + 4`).

**Ví dụ 3:** `"a"`, `k = 5` → `0` (chỉ một dòng — là dòng cuối).

**Ràng buộc:** `1 ≤ len(sentence) ≤ 5000`; `1 ≤ k ≤ 5000`; mỗi từ dài tối đa `k`; chỉ chữ thường và khoảng trắng.

*(Đề Premium — đã diễn đạt lại, hãy đối chiếu lại đề gốc.)*

In [1]:
import random
import time
from functools import lru_cache
from itertools import accumulate, product
from typing import List

examples = [(("i love leetcode", 12), 36), (("apples and bananas taste great", 7), 21), (("a", 5), 0)]
for (s, k), exp in examples:
    print(f"{s!r}, k={k} -> kỳ vọng {exp}")

'i love leetcode', k=12 -> kỳ vọng 36
'apples and bananas taste great', k=7 -> kỳ vọng 21
'a', k=5 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Độ dài một dòng

Dòng chứa các từ `i..j` có độ dài `Σ_{t=i}^{j} len(w_t) + (j − i)` (mỗi cặp từ kề nhau một khoảng trắng). Với tổng tiền tố `pre[t] = Σ_{x<t} len(w_x)`: `L(i, j) = pre[j+1] − pre[i] + (j − i)`.

In [2]:
words = "i love leetcode".split()
pre = list(accumulate((len(w) for w in words), initial=0))
L = lambda i, j: pre[j + 1] - pre[i] + (j - i)
print("độ dài từ:", [len(w) for w in words], "| L(0,1) =", L(0, 1), "| L(1,2) =", L(1, 2), "| L(0,2) =", L(0, 2))

độ dài từ: [1, 4, 8] | L(0,1) = 6 | L(1,2) = 13 | L(0,2) = 15


### 3.2 Quy hoạch động trên hậu tố

Đặt `dp[i]` = chi phí nhỏ nhất xếp các từ `i, i+1, …, m−1` **bắt đầu ở một dòng mới**. Dòng đầu của hậu tố chứa các từ `i..j`; phần còn lại là bài con `dp[j+1]`. Bài toán gốc là `dp[0]`.

### 3.3 Dòng cuối miễn phí

Nếu mọi từ `i..m−1` vừa trên một dòng (`L(i, m−1) ≤ k`) thì có thể đặt chúng làm **dòng cuối** với chi phí 0 ⇒ `dp[i] = 0` (không thể âm). Ngược lại dòng đầu của hậu tố **không** phải dòng cuối và phải trả `(k − L(i, j))²`.

### 3.4 Số cách ngắt

`m` từ có `m − 1` chỗ có thể ngắt ⇒ `2^(m−1)` cách (trước khi lọc độ dài) — vét cạn chỉ khả thi với `m` nhỏ.

## 4. Ý tưởng nền tảng / trực giác

Chi phí **bình phương** phạt nặng dòng rất trống ⇒ nên chia khoảng trống **đều** giữa các dòng hơn là để một dòng gần đầy và một dòng gần trống. Đó là lý do cần DP thay vì tham lam.

Tính tay **Ví dụ 1** (`k = 12`, độ dài từ `1, 4, 8`):

| i | hậu tố | vừa một dòng? | lựa chọn dòng đầu | dp[i] |
|---|---|---|---|---|
| 2 | `leetcode` | `8 ≤ 12` ✓ | — | 0 |
| 1 | `love leetcode` | `13 > 12` ✗ | `love` (4): `64 + dp[2] = 64` | 64 |
| 0 | `i love leetcode` | `15 > 12` ✗ | `i`: `121 + dp[1] = 185`; `i love` (6): `36 + dp[2] = 36` | **36** |

In [3]:
words, k = "i love leetcode".split(), 12
m = len(words); pre = list(accumulate((len(w) for w in words), initial=0))
dp = [0] * (m + 1)
for i in range(m - 1, -1, -1):
    if pre[m] - pre[i] + (m - 1 - i) <= k:
        dp[i] = 0; print(f"dp[{i}] = 0 (phần còn lại vừa một dòng)"); continue
    opts = []
    for j in range(i, m):
        Lij = pre[j + 1] - pre[i] + (j - i)
        if Lij > k: break
        opts.append(((k - Lij) ** 2 + dp[j + 1], words[i:j + 1]))
    dp[i] = min(o[0] for o in opts); print(f"dp[{i}] = {dp[i]} từ các lựa chọn {opts}")

dp[2] = 0 (phần còn lại vừa một dòng)
dp[1] = 64 từ các lựa chọn [(64, ['love'])]
dp[0] = 36 từ các lựa chọn [(185, ['i']), (36, ['i', 'love'])]


## 5. Cách 1 — Brute Force / baseline: thử mọi cách ngắt

Mỗi trong `m − 1` khe giữa hai từ hoặc ngắt hoặc không. Với mỗi mẫu, dựng các dòng, loại nếu dòng nào dài hơn `k`, cộng chi phí các dòng trừ dòng cuối.

In [4]:
def cost_bruteforce(sentence, k):
    w = sentence.split(); m = len(w); best = None
    for cuts in product((0, 1), repeat=m - 1):
        rows, cur = [], [w[0]]
        for t in range(1, m):
            if cuts[t - 1]:
                rows.append(cur); cur = [w[t]]
            else:
                cur.append(w[t])
        rows.append(cur)
        lens = [len(" ".join(r)) for r in rows]
        if max(lens) > k:
            continue
        c = sum((k - x) ** 2 for x in lens[:-1])
        best = c if best is None else min(best, c)
    return best

for (s, k), exp in examples:
    got = cost_bruteforce(s, k); print(got, "kỳ vọng", exp); assert got == exp

36 kỳ vọng 36
21 kỳ vọng 21
0 kỳ vọng 0


**Độ phức tạp:** `O(2^(m−1) · m)` — `m` tới ~2500 từ ⇒ bất khả thi. Dùng để kiểm chứng với `m ≤ 14`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cấu trúc con tối ưu):** sau khi chọn dòng đầu chứa `i..j`, phần còn lại là bài toán **độc lập** cùng dạng bắt đầu ở từ `j + 1` ⇒ `dp[i] = min_j ((k − L(i, j))² + dp[j + 1])` với `L(i, j) ≤ k`.

**Insight 2 (dòng cuối):** `dp[i] = 0` nếu `L(i, m−1) ≤ k`. Không cần trường hợp `j = m − 1` trong công thức trên (vì nếu dòng đầu chứa hết thì nó là dòng cuối).

**Insight 3 (giới hạn vòng trong):** độ dài dòng tăng theo `j`, dừng khi vượt `k`; một dòng chứa tối đa `⌊(k+1)/2⌋` từ ⇒ `O(m · min(m, k))`.

**Invariant (bottom-up):** khi tính `dp[i]`, mọi `dp[j]` với `j > i` đã đúng.

### 6.1 Minh họa: tham lam nhồi đầy dòng không tối ưu

Với `"aaa bb cc ddddd"`, `k = 6` (ví dụ tự dựng, đối chiếu lại đề gốc): tham lam xếp `"aaa bb" | "cc" | "ddddd"` (chi phí `0 + 16 = 16`), còn tối ưu là `"aaa" | "bb cc" | "ddddd"` (`9 + 1 = 10`).

In [5]:
def greedy_rows(sentence, k):
    rows, cur = [], ""
    for w in sentence.split():
        if not cur: cur = w
        elif len(cur) + 1 + len(w) <= k: cur += " " + w
        else: rows.append(cur); cur = w
    rows.append(cur)
    return rows, sum((k - len(r)) ** 2 for r in rows[:-1])

s, k = "aaa bb cc ddddd", 6
print("tham lam:", greedy_rows(s, k), "| tối ưu (vét cạn):", cost_bruteforce(s, k))

tham lam: (['aaa bb', 'cc', 'ddddd'], 16) | tối ưu (vét cạn): 10


## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP top-down có ghi nhớ

In [6]:
def cost_memo(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w)
    pre = list(accumulate(w, initial=0))
    @lru_cache(maxsize=None)
    def dp(i):
        if pre[m] - pre[i] + (m - 1 - i) <= k:
            return 0
        best = float("inf"); j = i
        while j < m:
            L = pre[j + 1] - pre[i] + (j - i)
            if L > k:
                break
            best = min(best, (k - L) ** 2 + dp(j + 1)); j += 1
        return best
    return dp(0)

print([cost_memo(s, k) for (s, k), _ in examples])

[36, 21, 0]


Đệ quy sâu tối đa `m` tầng (mỗi dòng một từ) — với `m ≈ 2500` vượt giới hạn mặc định 1000 của Python. Bản bottom-up bên dưới an toàn hơn.

**Độ phức tạp Cách 2:** `O(m · W)` với `W` = số từ tối đa trên một dòng.

### 7.2 Cách 3 — DP bottom-up (bản nộp)

In [7]:
class Solution:
    def minimumCost(self, sentence: str, k: int) -> int:
        w = [len(x) for x in sentence.split()]
        m = len(w)
        dp = [0] * (m + 1)
        suffix = -1                                  # độ dài nếu đặt w[i..m-1] trên một dòng
        for i in range(m - 1, -1, -1):
            suffix += w[i] + 1
            if suffix <= k:
                dp[i] = 0                            # dòng cuối, miễn phí
                continue
            best = float("inf")
            L = -1
            for j in range(i, m):
                L += w[j] + 1                        # độ dài dòng chứa w[i..j]
                if L > k:
                    break
                c = (k - L) ** 2 + dp[j + 1]
                if c < best:
                    best = c
            dp[i] = best
        return dp[0]

print([Solution().minimumCost(s, k) for (s, k), _ in examples])

[36, 21, 0]


Mẹo `L = −1` rồi cộng `w[j] + 1` mỗi từ: sau `t` từ, `L = Σ len − 1 + t` = tổng độ dài + `(t − 1)` khoảng trắng.

**Độ phức tạp Cách 3:** `O(m · W)` thời gian, `O(m)` bộ nhớ.

### 7.3 Cách 4 — DP tiền tố (dp theo **điểm kết thúc** dòng)

Đặt `f[j]` = chi phí nhỏ nhất xếp các từ `0..j−1` mà từ `j − 1` **kết thúc một dòng** (dòng đó đã bị tính phí). Đáp án = `min f[i]` qua các `i` sao cho các từ `i..m−1` vừa một dòng cuối. Cách nhìn này hữu ích khi dòng cuối được định nghĩa ở **đầu** câu thay vì cuối.

In [8]:
def cost_prefix(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w)
    INF = float("inf"); f = [INF] * (m + 1); f[0] = 0
    for j in range(1, m + 1):
        L = -1
        for i in range(j - 1, -1, -1):              # dòng chứa w[i..j-1]
            L += w[i] + 1
            if L > k: break
            f[j] = min(f[j], f[i] + (k - L) ** 2)
    ans, L = INF, -1
    for i in range(m - 1, -1, -1):                   # dòng cuối w[i..m-1]
        L += w[i] + 1
        if L > k: break
        ans = min(ans, f[i])
    return ans

print([cost_prefix(s, k) for (s, k), _ in examples])

[36, 21, 0]


## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** (`k = 7`, độ dài `6, 3, 7, 5, 5`):

In [9]:
def trace(sentence, k):
    words = sentence.split(); w = [len(x) for x in words]; m = len(w)
    dp = [0] * (m + 1); suffix = -1
    for i in range(m - 1, -1, -1):
        suffix += w[i] + 1
        if suffix <= k:
            print(f"i={i} ({words[i]}): hậu tố dài {suffix} <= {k} -> dp = 0"); continue
        best, L, opts = float("inf"), -1, []
        for j in range(i, m):
            L += w[j] + 1
            if L > k: break
            c = (k - L) ** 2 + dp[j + 1]; opts.append((" ".join(words[i:j + 1]), L, c)); best = min(best, c)
        dp[i] = best
        print(f"i={i} ({words[i]}): lựa chọn {opts} -> dp = {best}")
    return dp[0]

assert trace("apples and bananas taste great", 7) == 21

i=4 (great): hậu tố dài 5 <= 7 -> dp = 0
i=3 (taste): lựa chọn [('taste', 5, 4)] -> dp = 4
i=2 (bananas): lựa chọn [('bananas', 7, 4)] -> dp = 4
i=1 (and): lựa chọn [('and', 3, 20)] -> dp = 20
i=0 (apples): lựa chọn [('apples', 6, 21)] -> dp = 21


| i | từ | lựa chọn dòng đầu | dp[i] |
|---|---|---|---|
| 4 | great | hậu tố vừa ⇒ 0 | 0 |
| 3 | taste | `taste great` dài 11 > 7; `taste`: `4 + 0` | 4 |
| 2 | bananas | `bananas`: `0 + 4` | 4 |
| 1 | and | `and`: `16 + 4` (`and bananas` dài 11) | 20 |
| 0 | apples | `apples`: `1 + 20` | **21** |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — DP bottom-up trên hậu tố.

In [10]:
class Solution:
    def minimumCost(self, sentence: str, k: int) -> int:
        w = [len(x) for x in sentence.split()]
        m = len(w)
        dp = [0] * (m + 1)
        suffix = -1
        for i in range(m - 1, -1, -1):
            suffix += w[i] + 1
            if suffix <= k:
                dp[i] = 0
                continue
            best = float("inf")
            L = -1
            for j in range(i, m):
                L += w[j] + 1
                if L > k:
                    break
                c = (k - L) ** 2 + dp[j + 1]
                if c < best:
                    best = c
            dp[i] = best
        return dp[0]

for (s, k), exp in examples:
    got = Solution().minimumCost(s, k); print(got, exp); assert got == exp

36 36
21 21
0 0


## 10. Vì sao thuật toán đúng

Gọi `OPT(i)` = chi phí nhỏ nhất xếp `w_i..w_{m−1}` từ đầu một dòng mới (dòng cuối miễn phí).

**Trường hợp vừa một dòng:** nếu `L(i, m−1) ≤ k`, cách "một dòng" có chi phí 0, mà chi phí luôn `≥ 0` ⇒ `OPT(i) = 0`.

**Trường hợp không vừa:** mọi cách xếp hợp lệ có dòng đầu chứa `w_i..w_j` với `j < m − 1` (vì không vừa hết) và `L(i, j) ≤ k`; dòng đó **không** phải dòng cuối ⇒ chịu phí `(k − L(i,j))²`. Phần còn lại là một cách xếp hợp lệ của hậu tố `j + 1`, chi phí `≥ OPT(j + 1)`, và có thể đạt đúng `OPT(j + 1)` (ghép tùy ý) ⇒ `OPT(i) = min_j ((k − L(i,j))² + OPT(j+1))`. Dừng vòng `j` khi `L > k` là đúng vì `L` tăng theo `j`.

**Tồn tại lời giải:** mỗi từ dài `≤ k` nên "mỗi từ một dòng" luôn hợp lệ ⇒ `dp` hữu hạn. Thứ tự tính `i` giảm dần bảo đảm `dp[j+1]` có sẵn. ∎

### 10.1 Kiểm chứng thực nghiệm

So DP với vét cạn trên nhiều câu ngẫu nhiên (`m ≤ 12`), đồng thời dựng lại **cách ngắt** từ DP và tính lại chi phí của nó.

In [11]:
def reconstruct(sentence, k):
    words = sentence.split(); w = [len(x) for x in words]; m = len(w)
    dp = [0] * (m + 1); choice = [m] * (m + 1); suffix = -1
    for i in range(m - 1, -1, -1):
        suffix += w[i] + 1
        if suffix <= k: dp[i] = 0; choice[i] = m; continue
        best, L = float("inf"), -1
        for j in range(i, m):
            L += w[j] + 1
            if L > k: break
            c = (k - L) ** 2 + dp[j + 1]
            if c < best: best, choice[i] = c, j + 1
        dp[i] = best
    rows, i = [], 0
    while i < m:
        rows.append(" ".join(words[i:choice[i]])); i = choice[i]
    return dp[0], rows

def rand_sentence(rng, m, maxw):
    return " ".join("".join(rng.choice("ab") for _ in range(rng.randint(1, maxw))) for _ in range(m))

rng = random.Random(1010)
for _ in range(400):
    k = rng.randint(1, 12); s = rand_sentence(rng, rng.randint(1, 11), k)
    c, rows = reconstruct(s, k)
    assert c == cost_bruteforce(s, k)
    assert " ".join(rows) == s and max(len(r) for r in rows) <= k
    assert sum((k - len(r)) ** 2 for r in rows[:-1]) == c
print("DP = vét cạn; cách ngắt dựng lại hợp lệ và đúng chi phí (400 câu)")

DP = vét cạn; cách ngắt dựng lại hợp lệ và đúng chi phí (400 câu)


## 11. Phân tích Time / Space Complexity

Gọi `m` = số từ (`≤ 2500`), `W` = số từ tối đa trên một dòng (`≤ ⌊(k+1)/2⌋`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^(m−1) · m)` | `O(m)` |
| 2. Memo top-down | `O(m · W)` | `O(m)` + stack |
| 3. Bottom-up hậu tố | `O(m · W)` | `O(m)` |
| 4. DP tiền tố | `O(m · W)` | `O(m)` |

In [12]:
rng = random.Random(2052)
cases = [(" ".join(["a"] * 2500), 5000), (" ".join(["a"] * 2500), 100), (rand_sentence(rng, 700, 6), 40)]
for name, f in [("bottom_up", Solution().minimumCost), ("prefix", cost_prefix)]:
    t = time.perf_counter(); r = [f(s, k) for s, k in cases]
    print(f"{name:<10}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")

bottom_up : [0, 49, 535] (11 ms)


prefix    : [0, 49, 535] (310 ms)


## 12. Bộ test

In [13]:
ALL = {"memo": cost_memo, "bottom_up": Solution().minimumCost, "prefix": cost_prefix}
def run_all(s, k, brute=True):
    res = {n: f(s, k) for n, f in ALL.items()}
    if brute:
        res["brute"] = cost_bruteforce(s, k)
    assert len(set(res.values())) == 1, (s, k, res)
    return res["bottom_up"]

for (s, k), exp in examples:
    assert run_all(s, k) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [(("aaa bb cc ddddd", 6), 10), (("a b", 1), 0), (("a b c", 1), 0), (("ab cd", 2), 0),
        (("ab c d", 3), 1), (("a bb ccc", 3), 5), (("aaaa", 4), 0)]
for (s, k), exp in edge:
    assert run_all(s, k) == exp, (s, k, run_all(s, k))
rng = random.Random(20520)
for _ in range(400):
    k = rng.randint(1, 15); run_all(rand_sentence(rng, rng.randint(1, 12), min(k, 6)), k)
for _ in range(10):
    k = rng.randint(5, 200); s = rand_sentence(rng, rng.randint(100, 900), min(k, 8)); run_all(s, k, brute=False)
print("Ví dụ, 7 edge case, 400 câu nhỏ (vét cạn), 10 câu dài: OK")

Ví dụ, 7 edge case, 400 câu nhỏ (vét cạn), 10 câu dài: OK

### 12.1 Kiểm tra tính chất

- Nếu cả câu vừa một dòng (`len(sentence) ≤ k`) thì đáp án là `0`.
- Tăng `k` **không** nhất thiết giảm chi phí (chi phí phụ thuộc `k − n`) — nhưng đáp án luôn `≤` chi phí của cách "mỗi từ một dòng".
- Đáp án `≤` chi phí của cách tham lam.

In [14]:
rng = random.Random(121)
f = Solution().minimumCost
for _ in range(500):
    k = rng.randint(1, 20); s = rand_sentence(rng, rng.randint(1, 30), min(k, 6)); a = f(s, k)
    if len(s) <= k: assert a == 0
    w = s.split(); assert a <= sum((k - len(x)) ** 2 for x in w[:-1])
    assert a <= greedy_rows(s, k)[1]
print("3 tính chất đúng trên 500 câu")

3 tính chất đúng trên 500 câu


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tính phí cả dòng cuối

In [15]:
def wrong_last_row(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w)
    dp = [0] * (m + 1)
    for i in range(m - 1, -1, -1):
        best, L = float("inf"), -1
        for j in range(i, m):
            L += w[j] + 1
            if L > k: break
            best = min(best, (k - L) ** 2 + dp[j + 1])        # SAI: dòng cuối cũng bị tính
        dp[i] = best
    return dp[0]

for (s, k), exp in examples:
    print(f"sai {wrong_last_row(s, k)} | đúng {exp}")

sai 52 | đúng 36
sai 25 | đúng 21
sai 16 | đúng 0


**Vì sao sai:** đề: tổng chi phí **trừ dòng cuối**. Ví dụ 3 (`"a"`, `k = 5`) bản sai trả `16`. **Sửa:** `dp[i] = 0` khi phần còn lại vừa một dòng.

### Lỗi 2 — Tham lam nhồi đầy từng dòng

In [16]:
for (s, k) in [("aaa bb cc ddddd", 6), ("i love leetcode", 12)]:    # câu đầu tự dựng (đối chiếu lại đề gốc)
    print(f"{s!r}: tham lam {greedy_rows(s, k)[1]} | đúng {Solution().minimumCost(s, k)} | vét cạn {cost_bruteforce(s, k)}")

'aaa bb cc ddddd': tham lam 16 | đúng 10 | vét cạn 10
'i love leetcode': tham lam 36 | đúng 36 | vét cạn 36


**Vì sao sai:** chi phí bình phương phạt nặng dòng trống nhiều; để dòng đầu đầy và dòng sau rất trống tệ hơn chia đều (mục 6.1). **Sửa:** DP.

### Lỗi 3 — Quên khoảng trắng giữa các từ

In [17]:
def wrong_no_spaces(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w); dp = [0] * (m + 1)
    for i in range(m - 1, -1, -1):
        if sum(w[i:]) <= k: dp[i] = 0; continue                 # SAI: thiếu khoảng trắng
        best, L = float("inf"), 0
        for j in range(i, m):
            L += w[j]
            if L > k: break
            best = min(best, (k - L) ** 2 + dp[j + 1])
        dp[i] = best
    return dp[0]

for (s, k), exp in examples:
    print(f"sai {wrong_no_spaces(s, k)} | đúng {exp}")

sai 49 | đúng 36
sai 21 | đúng 21
sai 0 | đúng 0


**Vì sao sai:** `"love leetcode"` dài 13 (có một khoảng trắng), không phải 12 ⇒ bản sai cho phép dòng quá dài. **Sửa:** cộng `j − i` khoảng trắng.

### Lỗi 4 — Đệ quy không ghi nhớ

In [18]:
def no_memo(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w)
    def go(i):
        if sum(w[i:]) + (m - 1 - i) <= k: return 0
        best, L = float("inf"), -1
        for j in range(i, m):
            L += w[j] + 1
            if L > k: break
            best = min(best, (k - L) ** 2 + go(j + 1))
        return best
    return go(0)

for m in (14, 18, 22):
    s = " ".join(["a"] * m); k = 7
    t = time.perf_counter(); r1 = no_memo(s, k); t1 = time.perf_counter() - t
    t = time.perf_counter(); r2 = Solution().minimumCost(s, k); t2 = time.perf_counter() - t
    print(f"m={m}: không memo {t1 * 1000:7.0f} ms | DP {t2 * 1000:.2f} ms | {r1} == {r2}")

m=14: không memo       0 ms | DP 0.01 ms | 0 == 0
m=18: không memo       5 ms | DP 0.01 ms | 0 == 0
m=22: không memo      78 ms | DP 0.04 ms | 0 == 0


**Vì sao không tốt:** cùng một `go(i)` bị gọi lại theo số mũ (giống Fibonacci). **Sửa:** `lru_cache` hoặc bottom-up.

### Lỗi 5 — Dùng chi phí tuyến tính `|k − n|` thay vì bình phương

In [19]:
def wrong_linear(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w); dp = [0] * (m + 1); suffix = -1
    for i in range(m - 1, -1, -1):
        suffix += w[i] + 1
        if suffix <= k: dp[i] = 0; continue
        best, L = float("inf"), -1
        for j in range(i, m):
            L += w[j] + 1
            if L > k: break
            best = min(best, (k - L) + dp[j + 1])                 # SAI: thiếu bình phương
        dp[i] = best
    return dp[0]

for (s, k), exp in examples:
    print(f"sai {wrong_linear(s, k)} | đúng {exp}")

sai 6 | đúng 36
sai 7 | đúng 21
sai 0 | đúng 0


**Vì sao sai:** đề định nghĩa chi phí `(k − n)²`. Với chi phí tuyến tính, tổng chi phí các dòng gần như chỉ phụ thuộc **số dòng**, nên lời giải tối ưu (và giá trị) khác hẳn. **Sửa:** `(k − L) ** 2`. Ghi chú: một cách viết **khác nhưng đúng** là bỏ kiểm tra "hậu tố vừa một dòng" và cho chi phí 0 khi dòng đầu chứa tới từ cuối (`j = m − 1`) — tương đương vì `dp ≥ 0`.

### Lỗi 6 — Dừng vòng `j` bằng `>=` (bỏ dòng dài đúng `k`)

In [20]:
def wrong_ge(sentence, k):
    w = [len(x) for x in sentence.split()]; m = len(w); dp = [0] * (m + 1); suffix = -1
    for i in range(m - 1, -1, -1):
        suffix += w[i] + 1
        if suffix < k: dp[i] = 0; continue                       # SAI: phải <=
        best, L = float("inf"), -1
        for j in range(i, m):
            L += w[j] + 1
            if L >= k: break                                     # SAI: phải >
            best = min(best, (k - L) ** 2 + dp[j + 1])
        dp[i] = best
    return dp[0]

for (s, k), exp in examples + [(("ab cd", 5), 0), (("aaaa", 4), 0)]:
    print(f"{s!r}, k={k}: sai {wrong_ge(s, k)} | đúng {Solution().minimumCost(s, k)}")

'i love leetcode', k=12: sai 36 | đúng 36
'apples and bananas taste great', k=7: sai inf | đúng 21
'a', k=5: sai 0 | đúng 0
'ab cd', k=5: sai 9 | đúng 0
'aaaa', k=4: sai inf | đúng 0


**Vì sao sai:** dòng dài **đúng** `k` là hợp lệ ("at most k"). Với `"aaaa"`, `k = 4`, từ duy nhất không vừa theo `<` và vòng `j` cũng bỏ qua ⇒ `inf`. **Sửa:** `≤ k` / dừng khi `> k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia dãy thành các đoạn liên tiếp, chi phí mỗi đoạn phụ thuộc đoạn đó" ⇒ **DP phân đoạn**: `dp[i] = min_j (cost(i, j) + dp[j+1])`.
- Có ràng buộc độ dài ⇒ vòng trong dừng sớm, độ phức tạp `O(m · W)`.
- Phần tử "đặc biệt" (dòng cuối miễn phí) ⇒ xử lý ở **điều kiện biên** của DP.
- Chi phí lồi (bình phương) ⇒ tham lam thường sai; luôn thử phản ví dụ bằng vét cạn.
- Chuỗi dài ⇒ chỉ làm việc với **độ dài từ**.

## 15. Biến thể của bài

1. **Word wrap kinh điển** (Knuth): chi phí `(k − n)³` hoặc bao gồm cả dòng cuối.
2. **68 Text Justification** (Hard): tham lam xếp dòng rồi căn đều khoảng trắng.
3. **Chi phí khác** cho dòng cuối (ví dụ tính `(k − n)` tuyến tính).
4. **1335 Minimum Difficulty of a Job Schedule**: DP phân đoạn với số đoạn cố định.
5. **Số dòng tối thiểu** để chứa câu — tham lam là tối ưu (chi phí mỗi dòng như nhau).

### 15.1 Code biến thể: tính phí cả dòng cuối với mũ `p` tùy ý, đối chiếu vét cạn

In [21]:
def cost_general(sentence, k, p=2, last_free=True):
    w = [len(x) for x in sentence.split()]; m = len(w); dp = [0] * (m + 1); suffix = -1
    for i in range(m - 1, -1, -1):
        suffix += w[i] + 1
        if last_free and suffix <= k: dp[i] = 0; continue
        best, L = float("inf"), -1
        for j in range(i, m):
            L += w[j] + 1
            if L > k: break
            last = j == m - 1
            best = min(best, (0 if (last and last_free) else (k - L) ** p) + dp[j + 1])
        dp[i] = best
    return dp[0]

def brute_general(sentence, k, p, last_free):
    w = sentence.split(); m = len(w); best = None
    for cuts in product((0, 1), repeat=m - 1):
        rows, cur = [], [w[0]]
        for t in range(1, m):
            if cuts[t - 1]: rows.append(cur); cur = [w[t]]
            else: cur.append(w[t])
        rows.append(cur); lens = [len(" ".join(r)) for r in rows]
        if max(lens) > k: continue
        c = sum((k - x) ** p for x in (lens[:-1] if last_free else lens))
        best = c if best is None else min(best, c)
    return best

rng = random.Random(15)
for _ in range(300):
    k = rng.randint(1, 12); s = rand_sentence(rng, rng.randint(1, 9), min(k, 5)); p = rng.choice([1, 2, 3]); lf = rng.random() < 0.5
    assert cost_general(s, k, p, lf) == brute_general(s, k, p, lf)
print("Biến thể mũ p / tính phí dòng cuối khớp vét cạn;", cost_general("i love leetcode", 12, 3, False))

Biến thể mũ p / tính phí dòng cuối khớp vét cạn; 280


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi mẫu ngắt | `O(2^m · m)` | `O(m)` | Kiểm chứng |
| 2. Memo | DP hậu tố đệ quy | `O(m·W)` | `O(m)` + stack | Viết nhanh |
| 3. Bottom-up | DP hậu tố lặp | `O(m·W)` | `O(m)` | **Bản nộp** |
| 4. DP tiền tố | Theo điểm kết thúc | `O(m·W)` | `O(m)` | Khi cần duyệt xuôi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("a b", 1)`; (b) `("a bb ccc", 3)`; (c) `("ab cd", 2)`; (d) `("aa a aa", 4)`.

**Bài 2 (tính tay):** Lập bảng `dp` cho `"aaa bb cc ddddd"`, `k = 6`.

**Bài 3 (điền TODO):** Hoàn thiện `row_len(w, i, j)` (độ dài dòng chứa từ `i..j`) bằng tổng tiền tố.

**Bài 4 (tự viết lại):** Viết `best_rows(sentence, k)` trả danh sách các dòng của một cách xếp tối ưu.

**Bài 5 (mini challenge):** Biến thể: số dòng **tối đa** là `r` (nếu không xếp được trong `r` dòng thì trả `−1`). Viết DP `dp[i][t]` (hậu tố `i`, còn `t` dòng), đối chiếu vét cạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def row_len(w, i, j):
    pre = list(accumulate(w, initial=0))
    # TODO: trả pre[j + 1] - pre[i] + (j - i)
    pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_rows(sentence, k):
    # TODO: DP như Cách 3, lưu choice[i] = j + 1 tốt nhất, rồi dựng lại các dòng
    pass

def cost_max_rows(sentence, k, r):
    # TODO: dp[i][t]: nếu hậu tố i vừa 1 dòng và t >= 1 -> 0; ngược lại min_j cost + dp[j+1][t-1]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: kiểm tra vừa một dòng. Mức 2: mỗi từ một dòng khi `k` nhỏ. Mức 3: (a) `0` (`"a" | "b"`: chỉ dòng đầu tính `(1−1)² = 0`); (b) `5` (`"a bb"` dài 4 > 3 nên chỉ có `"a" | "bb" | "ccc"`: `4 + 1`); (c) `0` (`"ab" | "cd"`); (d) `0` (`"aa a"` dài đúng 4, dòng cuối `"aa"`).

**Bài 2.** Mức 1: độ dài `3, 2, 2, 5`. Mức 2: `dp[3] = 0`; `dp[2]`: `cc` (`16 + 0`) ⇒ 16; `dp[1]`: `bb` (`16 + 16 = 32`) hoặc `bb cc` (`1 + 0 = 1`) ⇒ 1. Mức 3: `dp[0]`: `aaa` (`9 + 1 = 10`), `aaa bb` (`0 + 16 = 16`) ⇒ `10`.

**Bài 3.** Mức 1: `pre[t]` = tổng `t` từ đầu. Mức 2: thêm `j − i` khoảng trắng. Mức 3: kiểm tra với `" ".join`.

**Bài 4.** Mức 1: lưu lựa chọn tốt nhất. Mức 2: khi hậu tố vừa một dòng, `choice[i] = m`. Mức 3: duyệt `i = choice[i]`.

**Bài 5.** Mức 1: thêm chiều "số dòng còn lại". Mức 2: `dp[m][t] = 0`, `dp[i][0] = inf` với `i < m`. Mức 3: `O(m · W · r)`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
f = Solution().minimumCost
b1 = [f(s, k) for s, k in [("a b", 1), ("a bb ccc", 3), ("ab cd", 2), ("aa a aa", 4)]]
print("Bài 1:", b1, "| vét cạn:", [cost_bruteforce(s, k) for s, k in [("a b", 1), ("a bb ccc", 3), ("ab cd", 2), ("aa a aa", 4)]])
assert b1 == [0, 5, 0, 0]
print("Bài 2:"); assert trace("aaa bb cc ddddd", 6) == 10
def row_len(w, i, j):
    pre = list(accumulate(w, initial=0))
    return pre[j + 1] - pre[i] + (j - i)
ws = "i love leetcode".split()
assert all(row_len([len(x) for x in ws], i, j) == len(" ".join(ws[i:j + 1])) for i in range(3) for j in range(i, 3))
print("Bài 3: OK")

Bài 1: [0, 5, 0, 0] | vét cạn: [0, 5, 0, 0]
Bài 2:
i=3 (ddddd): hậu tố dài 5 <= 6 -> dp = 0
i=2 (cc): lựa chọn [('cc', 2, 16)] -> dp = 16
i=1 (bb): lựa chọn [('bb', 2, 32), ('bb cc', 5, 1)] -> dp = 1
i=0 (aaa): lựa chọn [('aaa', 3, 10), ('aaa bb', 6, 16)] -> dp = 10
Bài 3: OK


In [25]:
# Bài 4, 5
def best_rows(sentence, k):
    return reconstruct(sentence, k)[1]

def cost_max_rows(sentence, k, r):
    w = [len(x) for x in sentence.split()]; m = len(w); INF = float("inf")
    dp = [[INF] * (r + 1) for _ in range(m + 1)]
    for t in range(r + 1): dp[m][t] = 0
    for i in range(m - 1, -1, -1):
        suffix = sum(w[i:]) + (m - 1 - i)
        for t in range(1, r + 1):
            if suffix <= k: dp[i][t] = 0; continue
            L = -1
            for j in range(i, m):
                L += w[j] + 1
                if L > k: break
                dp[i][t] = min(dp[i][t], (k - L) ** 2 + dp[j + 1][t - 1])
    return -1 if dp[0][r] == INF else dp[0][r]

def brute_max_rows(sentence, k, r):
    w = sentence.split(); m = len(w); best = None
    for cuts in product((0, 1), repeat=m - 1):
        if sum(cuts) + 1 > r: continue
        rows, cur = [], [w[0]]
        for t in range(1, m):
            if cuts[t - 1]: rows.append(cur); cur = [w[t]]
            else: cur.append(w[t])
        rows.append(cur); lens = [len(" ".join(x)) for x in rows]
        if max(lens) > k: continue
        c = sum((k - x) ** 2 for x in lens[:-1]); best = c if best is None else min(best, c)
    return -1 if best is None else best

rng = random.Random(5)
for _ in range(300):
    k = rng.randint(1, 12); s = rand_sentence(rng, rng.randint(1, 9), min(k, 5)); r = rng.randint(1, 6)
    rows = best_rows(s, k); assert sum((k - len(x)) ** 2 for x in rows[:-1]) == Solution().minimumCost(s, k)
    assert cost_max_rows(s, k, r) == brute_max_rows(s, k, r)
print("Bài 4, 5: OK |", best_rows("i love leetcode", 12), cost_max_rows("apples and bananas taste great", 7, 4))

Bài 4, 5: OK | ['i love', 'leetcode'] -1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (DP hậu tố):**

```python
dp = [0] * (m + 1); suffix = -1
for i in range(m - 1, -1, -1):
    suffix += w[i] + 1
    if suffix <= k: dp[i] = 0; continue          # dòng cuối miễn phí
    best, L = inf, -1
    for j in range(i, m):
        L += w[j] + 1                            # gồm khoảng trắng
        if L > k: break
        best = min(best, (k - L) ** 2 + dp[j + 1])
    dp[i] = best
return dp[0]
```

**Checklist:**
- [ ] Không tính phí dòng cuối?
- [ ] Độ dài dòng có tính **khoảng trắng**?
- [ ] Dòng dài **đúng** `k` là hợp lệ (`≤`)?
- [ ] DP có ghi nhớ / bottom-up (không đệ quy mũ, không quá sâu)?
- [ ] Không dùng tham lam?